# LeetCode 1277 — Count Square Submatrices with All Ones bằng Python

Link đề: https://leetcode.com/problems/count-square-submatrices-with-all-ones/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Matrix

Skeleton LeetCode:

```python
class Solution:
    def countSquares(self, matrix: list[list[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biết **liệt kê mọi hình vuông con** của một ma trận (góc + cạnh) và đếm chúng bằng vét cạn.
2. Dùng **prefix sum 2D** để kiểm tra "một hình vuông toàn số 1" trong `O(1)`.
3. Hiểu công thức DP kinh điển `dp[i][j] = 1 + min(trên, trái, chéo)` và ý nghĩa: **`dp[i][j]` vừa là cạnh lớn nhất, vừa là số hình vuông có góc dưới-phải tại `(i, j)`**.
4. Tối ưu bộ nhớ DP từ `O(mn)` xuống `O(n)` (một hàng + biến lưu ô chéo) và biết bẫy khi cập nhật tại chỗ.
5. Nhận dạng họ bài "hình vuông/hình chữ nhật toàn 1": LC 221, 1277, 1504, 85.

## 2. Đề bài & ví dụ

Cho ma trận `m × n` chỉ gồm 0 và 1. Hãy đếm **số hình vuông con** mà **mọi ô đều bằng 1**. Hình vuông cạnh 1 (một ô) cũng tính.

**Ví dụ 1:**
```
matrix = [[0,1,1,1],
          [1,1,1,1],
          [0,1,1,1]]
```
Kết quả `15`: 10 hình vuông cạnh 1, 4 hình vuông cạnh 2, 1 hình vuông cạnh 3.

**Ví dụ 2:**
```
matrix = [[1,0,1],
          [1,1,0],
          [1,1,0]]
```
Kết quả `7`: 6 hình vuông cạnh 1, 1 hình vuông cạnh 2.

**Ràng buộc:** `1 <= m, n <= 300`, `0 <= matrix[i][j] <= 1`.

In [1]:
import random
import time
from typing import List

EX1 = [[0, 1, 1, 1],
       [1, 1, 1, 1],
       [0, 1, 1, 1]]
EX2 = [[1, 0, 1],
       [1, 1, 0],
       [1, 1, 0]]
examples = [(EX1, 15), (EX2, 7)]

def show(mat):
    for row in mat:
        print(" ".join(str(x) for x in row))

for mat, want in examples:
    show(mat)
    print("-> mong đợi", want, "\n")

0 1 1 1
1 1 1 1
0 1 1 1
-> mong đợi 15 

1 0 1
1 1 0
1 1 0
-> mong đợi 7 



## 3. Kiến thức nền

### 3.1 Ma trận là list các list

`matrix[i][j]` là ô hàng `i`, cột `j`. `len(matrix)` là số hàng `m`, `len(matrix[0])` là số cột `n`.

In [2]:
m, n = len(EX1), len(EX1[0])
print("m =", m, "| n =", n)
print("ô (1, 2) =", EX1[1][2])
print("hàng 0 =", EX1[0], "| cột 0 =", [EX1[i][0] for i in range(m)])

m = 3 | n = 4
ô (1, 2) = 1
hàng 0 = [0, 1, 1, 1] | cột 0 = [0, 1, 0]


### 3.2 Tạo ma trận 0 đúng cách

`[[0] * n for _ in range(m)]` tạo `m` hàng **độc lập**. Còn `[[0] * n] * m` tạo `m` tham chiếu tới **cùng một** hàng — sửa một ô là sửa cả cột!

In [3]:
good = [[0] * 3 for _ in range(2)]
bad = [[0] * 3] * 2
good[0][0] = 9
bad[0][0] = 9
print("good:", good)
print("bad :", bad, "<- cả hai hàng đều đổi")

good: [[9, 0, 0], [0, 0, 0]]
bad : [[9, 0, 0], [9, 0, 0]] <- cả hai hàng đều đổi


### 3.3 Liệt kê hình vuông con

Một hình vuông con xác định bởi **góc trên-trái** `(r, c)` và **cạnh** `k`, với `r + k <= m` và `c + k <= n`. Số hình vuông con (bất kể giá trị) của ma trận `m × n` là `Σ_k (m−k+1)(n−k+1)`.

In [4]:
def all_squares(m, n):
    for k in range(1, min(m, n) + 1):
        for r in range(m - k + 1):
            for c in range(n - k + 1):
                yield r, c, k

sq = list(all_squares(3, 4))
print("số hình vuông con của lưới 3x4:", len(sq))
print("vài hình đầu (r, c, k):", sq[:5], "... cuối:", sq[-2:])

số hình vuông con của lưới 3x4: 20
vài hình đầu (r, c, k): [(0, 0, 1), (0, 1, 1), (0, 2, 1), (0, 3, 1), (1, 0, 1)] ... cuối: [(0, 0, 3), (0, 1, 3)]


### 3.4 Prefix sum 2D

`P[i+1][j+1]` = tổng các ô trong hình chữ nhật từ `(0, 0)` đến `(i, j)`:

```
P[i+1][j+1] = matrix[i][j] + P[i][j+1] + P[i+1][j] − P[i][j]
```

Tổng hình chữ nhật hàng `r1..r2`, cột `c1..c2` (bao gồm hai đầu):

```
P[r2+1][c2+1] − P[r1][c2+1] − P[r2+1][c1] + P[r1][c1]
```

In [5]:
def prefix2d(mat):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = mat[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return P

def rect_sum(P, r1, c1, r2, c2):
    return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]

P = prefix2d(EX1)
print("tổng cả ma trận:", rect_sum(P, 0, 0, 2, 3))
print("tổng hình vuông 2x2 tại (1,1):", rect_sum(P, 1, 1, 2, 2), "(= 4 nghĩa là toàn 1)")

tổng cả ma trận: 10
tổng hình vuông 2x2 tại (1,1): 4 (= 4 nghĩa là toàn 1)


### 3.5 `min` của nhiều giá trị và `sum` lồng nhau

`min(a, b, c)` lấy nhỏ nhất của ba số. `sum(map(sum, dp))` cộng toàn bộ ma trận.

In [6]:
print(min(3, 1, 2))
dp_demo = [[1, 2], [0, 3]]
print(sum(map(sum, dp_demo)), sum(sum(row) for row in dp_demo))

1
6 6


## 4. Ý tưởng nền tảng / trực giác

Đếm tay **Ví dụ 2**:

```
1 0 1
1 1 0
1 1 0
```

- Cạnh 1: đếm số ô bằng 1 ⇒ `1+1+1+1+1+1 = 6` (các ô (0,0), (0,2), (1,0), (1,1), (2,0), (2,1)).
- Cạnh 2: thử 4 vị trí góc trên-trái: (0,0) có ô (0,1)=0 ✗; (0,1) ✗; (1,0): ô (1,0),(1,1),(2,0),(2,1) đều 1 ✔; (1,1): có (1,2)=0 ✗. ⇒ 1.
- Cạnh 3: chỉ một vị trí, chứa 0 ✗.
- Tổng `6 + 1 = 7`. ✔

Một góc nhìn khác dẫn tới DP: **đếm theo góc dưới-phải**. Với mỗi ô `(i, j)`, hỏi "có bao nhiêu hình vuông toàn 1 **kết thúc** ở `(i, j)`?". Nếu hình vuông lớn nhất kết thúc ở đó có cạnh `s`, thì các hình vuông cạnh `1, 2, …, s` cũng kết thúc ở đó ⇒ đúng `s` hình. Đáp án = tổng các `s`.

In [7]:
# Đếm theo cạnh bằng vét cạn cho Ví dụ 1 để thấy 10 + 4 + 1
def is_all_ones(mat, r, c, k):
    return all(mat[i][j] == 1 for i in range(r, r + k) for j in range(c, c + k))

for mat, want in examples:
    m, n = len(mat), len(mat[0])
    by_size = {}
    for r, c, k in all_squares(m, n):
        if is_all_ones(mat, r, c, k):
            by_size[k] = by_size.get(k, 0) + 1
    print("theo cạnh:", by_size, "| tổng =", sum(by_size.values()), "| mong đợi", want)

theo cạnh: {1: 10, 2: 4, 3: 1} | tổng = 15 | mong đợi 15
theo cạnh: {1: 6, 2: 1} | tổng = 7 | mong đợi 7


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mọi góc trên-trái `(r, c)` và mọi cạnh `k`, kiểm tra từng ô trong hình vuông.

Cải tiến nhỏ: với cùng `(r, c)`, nếu hình vuông cạnh `k` đã có số 0 thì mọi hình vuông lớn hơn cũng có số 0 ⇒ `break`.

In [8]:
def count_brute(mat):
    m, n = len(mat), len(mat[0])
    total = 0
    for r in range(m):
        for c in range(n):
            for k in range(1, min(m - r, n - c) + 1):
                if is_all_ones(mat, r, c, k):
                    total += 1
                else:
                    break           # hình lớn hơn chứa hình này -> cũng có số 0
    return total

for mat, want in examples:
    print(count_brute(mat), "| mong đợi", want)

15 | mong đợi 15
7 | mong đợi 7


**Độ phức tạp:** có `O(mn·L)` cặp (góc, cạnh) với `L = min(m, n)`; mỗi lần kiểm tra tốn `O(k²)` ⇒ worst `O(mn·L³)`. Với ma trận toàn 1 cỡ `300 × 300` là khoảng `10^11` phép — quá chậm.

**Nút thắt:** khi tăng `k` lên `k+1`, ta kiểm tra lại cả `k²` ô cũ. Hai hướng sửa: (a) kiểm tra tổng hình vuông trong `O(1)` bằng prefix sum; (b) tái sử dụng kết quả của các ô lân cận (DP).

In [9]:
# Thấy nút thắt: ma trận toàn 1 kích thước nhỏ đã chậm dần rất nhanh
for size in [10, 20, 30]:
    mat = [[1] * size for _ in range(size)]
    t = time.perf_counter()
    res = count_brute(mat)
    print(f"{size}x{size}: {res} hình vuông, {time.perf_counter() - t:.3f}s")

10x10: 385 hình vuông, 0.001s
20x20: 2870 hình vuông, 0.014s
30x30: 9455 hình vuông, 0.084s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `dp[i][j]` = cạnh của **hình vuông toàn 1 lớn nhất có góc dưới-phải tại `(i, j)`**.

- Nếu `matrix[i][j] == 0` ⇒ `dp[i][j] = 0`.
- Nếu `matrix[i][j] == 1` và ở hàng 0 hoặc cột 0 ⇒ `dp[i][j] = 1`.
- Ngược lại ⇒ `dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])`.

**Vì sao là `min` của ba ô?** Hình vuông cạnh `s` kết thúc tại `(i, j)` gồm: ô `(i, j)` + hình vuông cạnh `s−1` kết thúc ở **trên** `(i−1, j)` + ở **trái** `(i, j−1)` + ở **chéo** `(i−1, j−1)`. Ba hình vuông này phủ kín toàn bộ phần còn lại. Muốn cả ba tồn tại thì `s − 1 ≤` cả ba giá trị ⇒ `s − 1 ≤ min(...)`.

**Quy tắc "nếu… thì…":** nếu `dp[i][j] = s` thì có đúng `s` hình vuông toàn 1 có góc dưới-phải tại `(i, j)` (cạnh 1…s) ⇒ **đáp án = Σ dp[i][j]**.

**Invariant:** khi duyệt theo hàng từ trên xuống, trong hàng từ trái sang, lúc tính `dp[i][j]` thì ba ô trên/trái/chéo đã có giá trị đúng.

## 7. Các cách tiếp theo

### Cách 2 — Prefix sum 2D + thử mọi hình vuông

**Từng đoạn:**
1. Dựng `P` (mục 3.4).
2. Với mỗi góc `(r, c)` và cạnh `k`: hình vuông toàn 1 ⇔ tổng bằng `k²`.
3. Vẫn dùng `break` khi gặp hình vuông không đạt.

In [10]:
def count_prefix(mat):
    m, n = len(mat), len(mat[0])
    P = prefix2d(mat)
    total = 0
    for r in range(m):
        for c in range(n):
            for k in range(1, min(m - r, n - c) + 1):
                if rect_sum(P, r, c, r + k - 1, c + k - 1) == k * k:
                    total += 1
                else:
                    break
    return total

for mat, want in examples:
    print(count_prefix(mat), "| mong đợi", want)

15 | mong đợi 15
7 | mong đợi 7


In [11]:
class SolutionPrefix:
    def countSquares(self, matrix: List[List[int]]) -> int:
        m, n = len(matrix), len(matrix[0])
        P = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                P[i + 1][j + 1] = matrix[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
        total = 0
        for r in range(m):
            for c in range(n):
                for k in range(1, min(m - r, n - c) + 1):
                    s = P[r + k][c + k] - P[r][c + k] - P[r + k][c] + P[r][c]
                    if s != k * k:
                        break
                    total += 1
        return total

print([SolutionPrefix().countSquares(mat) for mat, _ in examples])

[15, 7]


**Độ phức tạp Cách 2:** `O(mn)` dựng prefix + `O(mn·L)` thử hình vuông ⇒ worst `O(mn·min(m, n))` ≈ `2.7·10^7` với `300 × 300` toàn 1 (chậm trong Python nhưng là cải thiện lớn). Bộ nhớ `O(mn)`.

### Cách 3 — DP 2D: `dp[i][j] = 1 + min(trên, trái, chéo)`

**Từng đoạn:**
1. `dp` cùng kích thước `matrix`, khởi tạo 0.
2. Duyệt từng ô; ô bằng 1 thì tính theo công thức (hàng/cột 0 thì bằng 1).
3. Cộng dồn `dp[i][j]` vào đáp án.

In [12]:
def count_dp(mat):
    m, n = len(mat), len(mat[0])
    dp = [[0] * n for _ in range(m)]
    total = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 1:
                if i == 0 or j == 0:
                    dp[i][j] = 1
                else:
                    dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
                total += dp[i][j]
    return total

for mat, want in examples:
    print(count_dp(mat), "| mong đợi", want)

15 | mong đợi 15
7 | mong đợi 7


In [13]:
class SolutionDP:
    def countSquares(self, matrix: List[List[int]]) -> int:
        m, n = len(matrix), len(matrix[0])
        dp = [[0] * (n + 1) for _ in range(m + 1)]   # thêm viền 0 để khỏi xét biên
        total = 0
        for i in range(m):
            for j in range(n):
                if matrix[i][j]:
                    dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                    total += dp[i + 1][j + 1]
        return total

print([SolutionDP().countSquares(mat) for mat, _ in examples])

[15, 7]


Mẹo **viền 0**: dùng `dp` kích thước `(m+1) × (n+1)` và lệch chỉ số 1 ⇒ hàng/cột 0 tự động có "trên/trái/chéo" bằng 0, công thức cho ra `1` mà không cần `if i == 0 or j == 0`.

**Độ phức tạp Cách 3:** `O(mn)` thời gian, `O(mn)` bộ nhớ.

### Cách 4 — DP một hàng `O(n)` bộ nhớ

**Ý tưởng:** `dp[i][j]` chỉ cần hàng `i−1` và ô bên trái trong hàng `i`. Dùng một mảng `row` độ dài `n+1`:
- Trước khi ghi đè `row[j+1]`, giá trị cũ của nó là "**trên**".
- `row[j]` (đã cập nhật ở bước trước) là "**trái**".
- "**Chéo**" là giá trị cũ của `row[j]` trước khi bị ghi đè ⇒ phải lưu vào biến `prev`.

In [14]:
def count_dp_1d(mat):
    m, n = len(mat), len(mat[0])
    row = [0] * (n + 1)
    total = 0
    for i in range(m):
        prev = 0                       # dp[i-1][j-1] cho j = 0 (viền)
        for j in range(n):
            up = row[j + 1]            # dp[i-1][j] (chưa bị ghi đè)
            if mat[i][j]:
                row[j + 1] = 1 + min(up, row[j], prev)
                total += row[j + 1]
            else:
                row[j + 1] = 0
            prev = up                  # ô "trên" hiện tại là ô "chéo" của bước sau
    return total

for mat, want in examples:
    print(count_dp_1d(mat), "| mong đợi", want)

15 | mong đợi 15
7 | mong đợi 7


In [15]:
class Solution1D:
    def countSquares(self, matrix: List[List[int]]) -> int:
        n = len(matrix[0])
        row = [0] * (n + 1)
        total = 0
        for r in matrix:
            prev = 0
            for j, v in enumerate(r):
                up = row[j + 1]
                row[j + 1] = 1 + min(up, row[j], prev) if v else 0
                total += row[j + 1]
                prev = up
        return total

print([Solution1D().countSquares(mat) for mat, _ in examples])

[15, 7]


**Độ phức tạp Cách 4:** `O(mn)` thời gian, `O(n)` bộ nhớ.

### Cách 5 — DP tại chỗ (ghi đè lên `matrix`)

Nếu được phép sửa input, có thể lưu `dp` ngay trong `matrix`: `matrix[i][j] += min(...)` khi `matrix[i][j] == 1`. Bộ nhớ thêm `O(1)`. Nhược điểm: **phá dữ liệu đầu vào** — trong code thực tế nên tránh, ở đây ta làm trên bản sao để demo.

In [16]:
def count_inplace(mat):
    a = [row[:] for row in mat]        # bản sao để không phá input của notebook
    m, n = len(a), len(a[0])
    for i in range(1, m):
        for j in range(1, n):
            if a[i][j]:
                a[i][j] += min(a[i - 1][j], a[i][j - 1], a[i - 1][j - 1])
    return sum(map(sum, a))

for mat, want in examples:
    print(count_inplace(mat), "| mong đợi", want)

15 | mong đợi 15
7 | mong đợi 7


## 8. TRACE từng bước

Bảng `dp` của **Ví dụ 1** (cùng vị trí với ma trận):

```
matrix          dp
0 1 1 1         0 1 1 1
1 1 1 1   ->    1 1 2 2
0 1 1 1         0 1 2 3
```

Ví dụ ô `(2, 3)`: trên `dp[1][3] = 2`, trái `dp[2][2] = 2`, chéo `dp[1][2] = 2` ⇒ `1 + 2 = 3` ⇒ có 3 hình vuông kết thúc ở đây (cạnh 1, 2, 3). Tổng các ô dp: `3 + 6 + 6 = 15`. ✔

In [17]:
def trace_dp(mat):
    m, n = len(mat), len(mat[0])
    dp = [[0] * n for _ in range(m)]
    total = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                print(f"({i},{j}) = 0 -> dp = 0")
                continue
            if i == 0 or j == 0:
                dp[i][j] = 1
                print(f"({i},{j}) = 1, ở biên -> dp = 1")
            else:
                up, left, diag = dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1]
                dp[i][j] = 1 + min(up, left, diag)
                print(f"({i},{j}) = 1, trên={up}, trái={left}, chéo={diag} -> dp = {dp[i][j]}")
            total += dp[i][j]
            print(f"        tổng tạm = {total}")
    print("bảng dp:")
    show(dp)
    return total

print("đáp án =", trace_dp(EX2))

(0,0) = 1, ở biên -> dp = 1
        tổng tạm = 1
(0,1) = 0 -> dp = 0
(0,2) = 1, ở biên -> dp = 1
        tổng tạm = 2
(1,0) = 1, ở biên -> dp = 1
        tổng tạm = 3
(1,1) = 1, trên=0, trái=1, chéo=1 -> dp = 1
        tổng tạm = 4
(1,2) = 0 -> dp = 0
(2,0) = 1, ở biên -> dp = 1
        tổng tạm = 5
(2,1) = 1, trên=1, trái=1, chéo=1 -> dp = 2
        tổng tạm = 7
(2,2) = 0 -> dp = 0
bảng dp:
1 0 1
1 1 0
1 2 0
đáp án = 7


In [18]:
def trace_dp_1d(mat):
    n = len(mat[0])
    row = [0] * (n + 1)
    total = 0
    for i, r in enumerate(mat):
        prev = 0
        for j, v in enumerate(r):
            up = row[j + 1]
            row[j + 1] = 1 + min(up, row[j], prev) if v else 0
            total += row[j + 1]
            prev = up
        print(f"sau hàng {i}: row = {row[1:]}, tổng = {total}")
    return total

print("đáp án =", trace_dp_1d(EX1))

sau hàng 0: row = [0, 1, 1, 1], tổng = 3
sau hàng 1: row = [1, 1, 2, 2], tổng = 9
sau hàng 2: row = [0, 1, 2, 3], tổng = 15
đáp án = 15


## 9. Code nộp LeetCode

Chọn DP một hàng (Cách 4) — `O(mn)` thời gian, `O(n)` bộ nhớ, không sửa input.

In [19]:
class Solution:
    def countSquares(self, matrix: List[List[int]]) -> int:
        n = len(matrix[0])
        row = [0] * (n + 1)        # row[j+1] = dp của hàng đang xét tại cột j
        total = 0
        for r in matrix:
            prev = 0               # dp hàng trước, cột j-1 (ô chéo)
            for j, v in enumerate(r):
                up = row[j + 1]
                if v:
                    row[j + 1] = 1 + min(up, row[j], prev)
                    total += row[j + 1]
                else:
                    row[j + 1] = 0
                prev = up
        return total

for mat, want in examples:
    got = Solution().countSquares(mat)
    print(got, "OK" if got == want else "SAI")

15 OK
7 OK


## 10. Vì sao thuật toán đúng

**Bổ đề:** với ô `(i, j)` bằng 1 (không ở biên), hình vuông cạnh `s` có góc dưới-phải `(i, j)` toàn 1 **khi và chỉ khi** ba hình vuông cạnh `s−1` có góc dưới-phải tại `(i−1, j)`, `(i, j−1)`, `(i−1, j−1)` đều toàn 1.

- (⇒) Ba hình vuông nhỏ nằm trọn trong hình vuông lớn nên đều toàn 1.
- (⇐) Hợp của ba hình vuông nhỏ cùng ô `(i, j)` phủ kín hình vuông lớn: mọi ô `(x, y)` của hình lớn, nếu `x < i` và `y < j` thì thuộc hình chéo; nếu `x = i, y < j` thì thuộc hình trái; nếu `x < i, y = j` thì thuộc hình trên; còn lại là `(i, j)`.

Vì "có hình vuông cạnh `s−1` kết thúc tại ô X" ⇔ `s − 1 ≤ dp[X]`, suy ra cạnh lớn nhất tại `(i, j)` là `1 + min(ba dp)`.

**Đếm đúng:** mỗi hình vuông toàn 1 có **đúng một** góc dưới-phải, nên được đếm đúng một lần ở ô đó; và tại ô đó, các cạnh `1..dp[i][j]` đều hợp lệ (hình vuông nhỏ hơn nằm trong hình vuông lớn nhất cùng góc). ⇒ Tổng `dp` = đáp án.

**Bản 1 hàng đúng** vì tại thời điểm tính cột `j`: `row[j+1]` còn giữ giá trị hàng trên (chưa ghi đè), `row[j]` là giá trị hàng hiện tại đã tính, `prev` giữ giá trị hàng trên ở cột `j−1` trước khi bị ghi đè.

## 11. Phân tích Time / Space

Gọi `m` = số hàng, `n` = số cột, `L = min(m, n)`.

| Cách | Time worst | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(mn·L³)` | `O(1)` | `break` giúp nhiều khi ma trận nhiều số 0 |
| 2. Prefix sum 2D | `O(mn·L)` | `O(mn)` | worst khi ma trận toàn 1 |
| 3. DP 2D | `O(mn)` | `O(mn)` | rõ ràng nhất |
| 4. DP 1 hàng | `O(mn)` | `O(n)` | **nộp bài** |
| 5. DP tại chỗ | `O(mn)` | `O(1)` thêm | sửa input |

Với `300 × 300 = 9·10^4` ô, DP chỉ cần ~`10^5` bước.

## 12. Bộ test

Ví dụ đề, edge case (1×1, một hàng, một cột, toàn 0, toàn 1, có công thức đóng), đối chiếu ngẫu nhiên (seed cố định) giữa mọi cách, và test hiệu năng `300 × 300`.

In [20]:
ALL = {
    "brute": count_brute, "prefix": count_prefix, "dp": count_dp,
    "dp1d": count_dp_1d, "inplace": count_inplace,
    "SolPrefix": lambda a: SolutionPrefix().countSquares(a),
    "SolDP": lambda a: SolutionDP().countSquares(a),
    "Sol1D": lambda a: Solution1D().countSquares(a),
    "Solution": lambda a: Solution().countSquares(a),
}

def full_ones_count(m, n):
    return sum((m - k + 1) * (n - k + 1) for k in range(1, min(m, n) + 1))

fixed = [
    (EX1, 15), (EX2, 7),
    ([[0]], 0), ([[1]], 1),
    ([[1, 1, 1, 1]], 4), ([[1], [1], [0]], 2),
    ([[0, 0], [0, 0]], 0), ([[1, 1], [1, 1]], 5),
    ([[1] * 4 for _ in range(3)], full_ones_count(3, 4)),
    ([[0, 1], [1, 1]], 3),             # ô chéo bằng 0 -> không có hình 2x2
]
for mat, want in fixed:
    for name, f in ALL.items():
        got = f(mat)
        assert got == want, (name, mat, got, want)
print("Test cố định: qua", len(fixed), "case; lưới 3x4 toàn 1 có", full_ones_count(3, 4), "hình vuông")

Test cố định: qua 10 case; lưới 3x4 toàn 1 có 20 hình vuông


In [21]:
rng = random.Random(1277)
for trial in range(400):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    p = rng.choice([0.5, 0.75, 0.9])          # mật độ số 1 khác nhau
    mat = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]
    snapshot = [row[:] for row in mat]
    want = count_brute(mat)
    for name, f in ALL.items():
        assert f(mat) == want, (name, mat)
    assert mat == snapshot, "không cách nào được phá input"
print("400 ma trận ngẫu nhiên: mọi cách khớp brute force, input không bị sửa")

400 ma trận ngẫu nhiên: mọi cách khớp brute force, input không bị sửa


In [22]:
# Hiệu năng: 300x300 toàn 1 (worst case) và ngẫu nhiên
big_ones = [[1] * 300 for _ in range(300)]
big_rand = [[1 if rng.random() < 0.8 else 0 for _ in range(300)] for _ in range(300)]
for name in ["dp", "dp1d", "Solution", "SolDP"]:
    t = time.perf_counter()
    a = ALL[name](big_ones)
    b = ALL[name](big_rand)
    print(f"{name:>8}: toàn 1 -> {a}, ngẫu nhiên -> {b}, {time.perf_counter() - t:.3f}s")
assert Solution().countSquares(big_ones) == full_ones_count(300, 300)

      dp: toàn 1 -> 9045050, ngẫu nhiên -> 123613, 0.040s
    dp1d: toàn 1 -> 9045050, ngẫu nhiên -> 123613, 0.021s


Solution: toàn 1 -> 9045050, ngẫu nhiên -> 123613, 0.021s
   SolDP: toàn 1 -> 9045050, ngẫu nhiên -> 123613, 0.029s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả về `max(dp)` (bài LC 221) thay vì `sum(dp)`

LC 221 *Maximal Square* hỏi diện tích hình vuông lớn nhất; bài này hỏi **số lượng**.

In [23]:
def wrong_max(mat):
    m, n = len(mat), len(mat[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                best = max(best, dp[i + 1][j + 1])
    return best                     # SAI: đây là cạnh lớn nhất

print("Code sai:", wrong_max(EX1), "| đúng là:", Solution().countSquares(EX1))

Code sai: 3 | đúng là: 15


**Code đúng:** cộng dồn `total += dp[i+1][j+1]` (như `SolutionDP`).

### Lỗi 2 — Chỉ lấy `min` của trên và trái, quên ô chéo

In [24]:
def wrong_no_diag(mat):
    m, n = len(mat), len(mat[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    total = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j])   # SAI
                total += dp[i + 1][j + 1]
    return total

bad = [[0, 1], [1, 1]]
print("Code sai:", wrong_no_diag(bad), "| đúng là:", Solution().countSquares(bad))

Code sai: 4 | đúng là: 3


Ô `(0,0) = 0` nằm ở góc chéo nên không có hình vuông 2×2, nhưng trên và trái đều bằng 1 ⇒ code sai đếm thêm 1. **Code đúng:** `min(trên, trái, chéo)`.

### Lỗi 3 — Tạo ma trận dp bằng `[[0] * n] * m`

In [25]:
def wrong_alias(mat):
    m, n = len(mat), len(mat[0])
    dp = [[0] * (n + 1)] * (m + 1)          # SAI: các hàng dùng chung một list
    total = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                total += dp[i + 1][j + 1]
            else:
                dp[i + 1][j + 1] = 0
    return total

print("Code sai:", wrong_alias(EX1), "| đúng là:", Solution().countSquares(EX1))

Code sai: 16 | đúng là: 15


Vì mọi hàng là **cùng một list**, "hàng trên" chính là "hàng hiện tại" ⇒ giá trị bị lẫn. **Code đúng:** `[[0] * (n + 1) for _ in range(m + 1)]`.

### Lỗi 4 — DP một hàng nhưng quên lưu ô chéo `prev`

In [26]:
def wrong_1d_no_prev(mat):
    n = len(mat[0])
    row = [0] * (n + 1)
    total = 0
    for r in mat:
        for j, v in enumerate(r):
            # SAI: row[j] lúc này đã là giá trị HÀNG HIỆN TẠI, không còn là ô chéo
            row[j + 1] = 1 + min(row[j + 1], row[j], row[j]) if v else 0
            total += row[j + 1]
    return total

bad2 = [[0, 1], [1, 1]]
print("Code sai:", wrong_1d_no_prev(bad2), "| đúng là:", Solution().countSquares(bad2))

Code sai: 4 | đúng là: 3


**Code đúng:** đọc `up = row[j+1]` trước khi ghi đè, dùng `prev` cho ô chéo rồi gán `prev = up` (như `Solution`).

### Lỗi 5 — Brute force kiểm tra "cạnh `k`" nhưng tính nhầm biên `range`

Viết `range(1, min(m - r, n - c))` (thiếu `+ 1`) bỏ sót hình vuông chạm biên phải/dưới.

In [27]:
def wrong_bound(mat):
    m, n = len(mat), len(mat[0])
    total = 0
    for r in range(m):
        for c in range(n):
            for k in range(1, min(m - r, n - c)):      # SAI: thiếu +1
                if is_all_ones(mat, r, c, k):
                    total += 1
                else:
                    break
    return total

print("Code sai:", wrong_bound(EX1), "| đúng là:", count_brute(EX1))
print("Code sai trên [[1]]:", wrong_bound([[1]]), "| đúng là:", count_brute([[1]]))

Code sai: 6 | đúng là: 15
Code sai trên [[1]]: 0 | đúng là: 1


**Code đúng:** `range(1, min(m - r, n - c) + 1)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm/tìm **hình vuông toàn 1**" ⇒ gần như chắc chắn là DP `1 + min(3 ô)`.
- Muốn **đếm** ⇒ cộng `dp`; muốn **cạnh lớn nhất** ⇒ `max(dp)`; muốn **diện tích lớn nhất** ⇒ `max(dp)²`.
- Thêm **viền 0** (`(m+1) × (n+1)`) để khỏi viết `if` cho biên.
- Cần kiểm tra tổng một vùng chữ nhật nhiều lần ⇒ **prefix sum 2D**.
- Tối ưu bộ nhớ một hàng: luôn hỏi "giá trị nào bị ghi đè trước khi mình cần?" ⇒ lưu vào biến tạm (`prev`).
- Ràng buộc `300 × 300` ⇒ `O(mn)` hoặc `O(mn·L)` (nếu ngôn ngữ nhanh) đều được; Python nên dùng `O(mn)`.

## 15. Biến thể của bài

1. **LC 221 Maximal Square:** diện tích hình vuông toàn 1 lớn nhất ⇒ `max(dp)²`.
2. **LC 1504 Count Submatrices With All Ones:** đếm **hình chữ nhật** toàn 1 — khó hơn, dùng "chiều cao cột" + duyệt hoặc stack đơn điệu.
3. **LC 85 Maximal Rectangle:** hình chữ nhật toàn 1 lớn nhất — histogram + stack.
4. **LC 1292 Maximum Side Length of a Square with Sum ≤ Threshold:** prefix sum 2D + binary search / tăng dần cạnh.
5. **Đếm hình vuông toàn cùng một giá trị** (không chỉ số 1): điều kiện thêm là 4 ô liên quan có cùng giá trị.
6. **Đếm hình vuông theo từng kích thước:** từ `dp[i][j] = s`, mỗi cạnh `1..s` được cộng 1 — dùng mảng hiệu.

In [28]:
def max_square_area(mat):           # LC 221
    m, n = len(mat), len(mat[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                best = max(best, dp[i + 1][j + 1])
    return best * best

print("Diện tích hình vuông lớn nhất Ví dụ 1:", max_square_area(EX1))
print("Diện tích hình vuông lớn nhất Ví dụ 2:", max_square_area(EX2))

Diện tích hình vuông lớn nhất Ví dụ 1: 9
Diện tích hình vuông lớn nhất Ví dụ 2: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi (góc, cạnh), kiểm tra từng ô | `O(mn·L³)` | `O(1)` | làm chuẩn để test |
| 2. Prefix sum 2D | tổng hình vuông = `k²` | `O(mn·L)` | `O(mn)` | khi điều kiện là "tổng vùng" tổng quát |
| 3. DP 2D | `1 + min(trên, trái, chéo)`, cộng dồn | `O(mn)` | `O(mn)` | dễ hiểu, dễ debug |
| 4. DP 1 hàng | như 3, giữ `prev` cho ô chéo | `O(mn)` | `O(n)` | **nộp bài**, tiết kiệm bộ nhớ |
| 5. DP tại chỗ | ghi dp vào `matrix` | `O(mn)` | `O(1)` | khi được phép sửa input |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** Đoán `countSquares` của `[[1,1],[1,1]]`, `[[1,1,1]]`, `[[1,1,0],[1,1,1],[0,1,1]]`.

**Bài 2 (tính tay):** Lập bảng `dp` cho
```
1 1 1
1 1 1
1 1 0
```
và tính tổng.

**Bài 3 (điền TODO):** Hoàn thiện `count_by_size_todo(mat)` trả về dict `{cạnh: số hình vuông}` chỉ dùng bảng `dp` (không vét cạn).

**Bài 4 (tự viết lại):** Tự viết lại DP **tại chỗ** nhưng trên **bản sao** của input, dùng viền 0 tưởng tượng (không có `if i == 0 or j == 0`, chỉ duyệt `i, j ≥ 1`).

**Bài 5 (mini challenge):** Viết `count_rectangles(mat)` đếm **số hình chữ nhật con** toàn 1 (LC 1504). Gợi ý: `O(m·n²)` là đủ cho ma trận nhỏ.

In [29]:
def count_by_size_todo(mat):
    m, n = len(mat), len(mat[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    res = {}
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                # TODO: với mỗi cạnh k = 1..dp[i+1][j+1], tăng res[k] thêm 1
    return res

# Lưu ý: cell này chỉ định nghĩa hàm, không gọi.

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đếm số ô bằng 1. (2) Tìm các khối 2×2 toàn 1. (3) Với ma trận 3×3 thứ ba: có hai khối 2×2 toàn 1 (góc trên-trái và góc dưới-phải); không có 3×3.

**Bài 2.** (1) Hàng 0 và cột 0 đều là 1. (2) `dp[1][1] = 1 + min(1,1,1) = 2`, `dp[1][2] = 2`, `dp[2][1] = 2`. (3) `dp[2][2] = 0` vì ô bằng 0 ⇒ tổng theo hàng `3 + 5 + 3`.

**Bài 3.** (1) `for k in range(1, s + 1): res[k] = res.get(k, 0) + 1`. (2) Hoặc đếm `cnt[s] += 1` rồi cộng dồn từ lớn xuống nhỏ. (3) So với vét cạn theo cạnh ở mục 4.

**Bài 4.** (1) `a = [row[:] for row in mat]`. (2) Duyệt `i` từ 1, `j` từ 1, `if a[i][j]: a[i][j] = 1 + min(...)`. (3) Đáp án `sum(map(sum, a))` — hàng/cột 0 giữ nguyên 0/1.

**Bài 5.** (1) Cố định hàng trên `r1`, mở rộng hàng dưới `r2`, duy trì `col_ok[j]` = cột `j` toàn 1 từ `r1` tới `r2`. (2) Với dãy `col_ok`, số đoạn liên tiếp toàn `True` có độ dài `L` đóng góp `L(L+1)/2`. (3) Kiểm chứng bằng vét cạn 4 vòng lặp.

## 19. Đáp án tham khảo

In [30]:
# Bài 1
assert Solution().countSquares([[1, 1], [1, 1]]) == 5
assert Solution().countSquares([[1, 1, 1]]) == 3
assert Solution().countSquares([[1, 1, 0], [1, 1, 1], [0, 1, 1]]) == 9

# Bài 2
assert Solution().countSquares([[1, 1, 1], [1, 1, 1], [1, 1, 0]]) == 11

# Bài 3
def count_by_size_sol(mat):
    m, n = len(mat), len(mat[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    res = {}
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                dp[i + 1][j + 1] = 1 + min(dp[i][j + 1], dp[i + 1][j], dp[i][j])
                for k in range(1, dp[i + 1][j + 1] + 1):
                    res[k] = res.get(k, 0) + 1
    return res

assert count_by_size_sol(EX1) == {1: 10, 2: 4, 3: 1}
assert count_by_size_sol(EX2) == {1: 6, 2: 1}

# Bài 4
def count_inplace_sol(mat):
    a = [row[:] for row in mat]
    for i in range(1, len(a)):
        for j in range(1, len(a[0])):
            if a[i][j]:
                a[i][j] = 1 + min(a[i - 1][j], a[i][j - 1], a[i - 1][j - 1])
    return sum(map(sum, a))

# Bài 5
def count_rectangles_sol(mat):
    m, n = len(mat), len(mat[0])
    total = 0
    for r1 in range(m):
        col_ok = [True] * n
        for r2 in range(r1, m):
            run = 0
            for j in range(n):
                col_ok[j] = col_ok[j] and mat[r2][j] == 1
                run = run + 1 if col_ok[j] else 0
                total += run          # số hình chữ nhật có cạnh phải tại cột j
    return total

def count_rectangles_brute(mat):
    m, n = len(mat), len(mat[0])
    return sum(1 for r1 in range(m) for r2 in range(r1, m)
               for c1 in range(n) for c2 in range(c1, n)
               if all(mat[i][j] for i in range(r1, r2 + 1) for j in range(c1, c2 + 1)))

assert count_rectangles_sol([[1, 0, 1], [1, 1, 0], [1, 1, 0]]) == 13   # ví dụ của LC 1504

rng = random.Random(77)
for _ in range(300):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    mat = [[rng.randint(0, 1) if rng.random() < 0.5 else 1 for _ in range(n)] for _ in range(m)]
    assert count_inplace_sol(mat) == count_brute(mat)
    assert sum(count_by_size_sol(mat).values()) == count_brute(mat)
    assert count_rectangles_sol(mat) == count_rectangles_brute(mat)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DP hình vuông (viền 0):**

```python
dp = [[0] * (n + 1) for _ in range(m + 1)]
total = 0
for i in range(m):
    for j in range(n):
        if matrix[i][j]:
            dp[i+1][j+1] = 1 + min(dp[i][j+1], dp[i+1][j], dp[i][j])
            total += dp[i+1][j+1]        # đếm  (max -> LC 221)
```

**Template 1 hàng:**

```python
row = [0] * (n + 1)
for r in matrix:
    prev = 0
    for j, v in enumerate(r):
        up = row[j+1]
        row[j+1] = 1 + min(up, row[j], prev) if v else 0
        prev = up
```

**Checklist trước khi submit:**
- [ ] Ô bằng 0 ⇒ `dp = 0` (đặc biệt khi dùng 1 hàng: phải ghi đè về 0).
- [ ] `min` của **ba** ô: trên, trái, chéo.
- [ ] Cộng dồn (`sum`), không phải `max`.
- [ ] Tạo ma trận bằng list comprehension, không dùng `[[0]*n]*m`.
- [ ] Bản 1 hàng: lưu `up` trước khi ghi đè, cập nhật `prev = up`.
- [ ] Đúng chữ ký: `countSquares(self, matrix: List[List[int]]) -> int`.